# Complementary Experiments Notebook
Standalone version (no custom src modules).

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans
from sklearn.utils import resample
import xgboost as xgb

In [ ]:
# 1. Load processed data
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").values.ravel()
y_test = pd.read_csv("y_test.csv").values.ravel()

In [ ]:
# 2. Undersampling
df_train = pd.concat([X_train, pd.Series(y_train, name="target")], axis=1)

majority = df_train[df_train.target == 0]
minority = df_train[df_train.target == 1]

majority_downsampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

df_balanced = pd.concat([majority_downsampled, minority])

X_train = df_balanced.drop("target", axis=1)
y_train = df_balanced["target"]

In [ ]:
# 3. Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

print("Random Forest Results:")
print(classification_report(y_test, rf_preds))

In [ ]:
# 4. XGBoost
xgb_model = xgb.XGBClassifier(eval_metric="logloss")
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

print("XGBoost Results:")
print(classification_report(y_test, xgb_preds))

In [ ]:
# 5. KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_train)

print("Cluster distribution:")
print(pd.Series(clusters).value_counts())